# Lesson 2.2 — Chart.yaml & values.yaml

**Phase 2: Chart Anatomy & Templates**

---

## Learning Objectives

By the end of this lesson you will be able to:
- Write a complete, well-structured `Chart.yaml`
- Design a `values.yaml` that supports customisation without breaking changes
- Override values at install time using `--set` and `-f`
- Understand value merging and the global values pattern

---

## Chart.yaml — Complete Field Reference

> **Reference:** [Charts Guide — Chart.yaml](https://helm.sh/docs/topics/charts/#the-chartyaml-file)

In [ ]:
%%writefile /tmp/full-Chart.yaml
apiVersion: v2                     # Required. Use v2 for Helm 3+.
name: mywebapp                     # Required. The chart name.
description: A production-ready web application chart
type: application                  # 'application' (default) or 'library'
version: 1.2.3                     # Required. SemVer. This is the CHART version.
appVersion: "2.0.1"                # The version of the APP inside the chart. Quote it!
kubeVersion: ">=1.25.0"            # Optional. Minimum compatible Kubernetes version.
keywords:
  - web
  - nginx
  - backend
home: https://example.com
sources:
  - https://github.com/example/mywebapp
maintainers:
  - name: Jane Smith
    email: jane@example.com
    url: https://example.com
icon: https://example.com/icon.svg
deprecated: false                  # Set to true when retiring a chart
annotations:
  category: WebServer              # Custom metadata goes in annotations

In [ ]:
cat /tmp/full-Chart.yaml

### Important Distinctions

| Field | Controls | Example |
|---|---|---|
| `version` | The chart package itself | `1.2.3` — bump when you change templates |
| `appVersion` | The app running inside | `"2.0.1"` — reflects your Docker image tag |

**Always quote `appVersion`** — YAML would interpret `1.0` as a float `1.0` and `1234e10` as scientific notation.

---

## values.yaml — Designing for Customisation

A well-designed `values.yaml` follows these principles:

1. **Flat for simple things, nested for complex ones** — Don't over-nest; keep it readable.
2. **Sensible defaults** — The chart should work with `helm install` and zero user config.
3. **Comments** — Explain every non-obvious value.
4. **Group related settings** — image, service, ingress, resources each get their own block.

In [ ]:
%%writefile /tmp/example-values.yaml
# Number of pod replicas to run
replicaCount: 1

# Container image settings
image:
  repository: nginx
  pullPolicy: IfNotPresent
  # Overrides the image tag; defaults to Chart.appVersion if blank
  tag: ""

imagePullSecrets: []
nameOverride: ""
fullnameOverride: ""

# Kubernetes Service settings
service:
  type: ClusterIP
  port: 80

# Ingress settings
ingress:
  enabled: false
  className: ""
  annotations: {}
  hosts:
    - host: chart-example.local
      paths:
        - path: /
          pathType: ImplementationSpecific
  tls: []

# Resource limits and requests
resources:
  limits:
    cpu: 100m
    memory: 128Mi
  requests:
    cpu: 50m
    memory: 64Mi

# Node selectors, tolerations, affinity
nodeSelector: {}
tolerations: []
affinity: {}

# Global values — visible to all subcharts
global:
  imageRegistry: ""
  storageClass: ""

---

## How Value Overriding Works

When you run `helm install`, Helm deep-merges values in this order:

```
chart's values.yaml (base defaults)
    ↑ merged with
-f custom-values.yaml (user file)
    ↑ merged with
--set key=value (individual overrides; highest priority)
```

In [ ]:
# Set up a working chart directory for demonstration
rm -rf /tmp/valsdemo && mkdir -p /tmp/valsdemo/templates

# Chart.yaml
cat > /tmp/valsdemo/Chart.yaml << 'EOF'
apiVersion: v2
name: valsdemo
version: 0.1.0
appVersion: "1.0"
EOF

# values.yaml — defaults
cat > /tmp/valsdemo/values.yaml << 'EOF'
env: development
replicaCount: 1
image:
  tag: "latest"
EOF

# A template that prints the values
cat > /tmp/valsdemo/templates/configmap.yaml << 'EOF'
apiVersion: v1
kind: ConfigMap
metadata:
  name: {{ .Release.Name }}-config
data:
  environment: {{ .Values.env }}
  replicas: {{ .Values.replicaCount | quote }}
  imageTag: {{ .Values.image.tag }}
EOF

echo "Chart ready"

In [ ]:
# 1. Render with chart defaults only
echo "=== Defaults only ==="
helm template myapp /tmp/valsdemo

echo ""

# 2. Override with a values file
cat > /tmp/prod-values.yaml << 'EOF'
env: production
replicaCount: 3
EOF

echo "=== With -f prod-values.yaml ==="
helm template myapp /tmp/valsdemo -f /tmp/prod-values.yaml

echo ""

# 3. --set overrides the file
echo "=== With -f prod-values.yaml AND --set replicaCount=5 ==="
helm template myapp /tmp/valsdemo -f /tmp/prod-values.yaml --set replicaCount=5

---

## Global Values

Values under the `global:` key are special — they are automatically passed to all sub-charts. This is the standard pattern for sharing config across a multi-chart deployment:

In [ ]:
%%writefile /tmp/global-values-example.yaml
# Parent chart values.yaml

global:
  imageRegistry: "my-registry.example.com"  # Available in ALL subcharts as .Values.global.imageRegistry
  storageClass: "fast-ssd"

# These are only visible to THIS chart:
replicaCount: 2

# Values for a subchart called 'postgresql':
postgresql:
  auth:
    password: "supersecret"   # DO NOT put real secrets in values.yaml!
                              # Use Kubernetes Secrets or a secrets manager instead.

In [ ]:
cat /tmp/global-values-example.yaml

> **Security Warning:** Never store passwords, tokens, or API keys in `values.yaml`. Use Kubernetes Secrets and reference them, or integrate a secrets manager like HashiCorp Vault or External Secrets Operator.

---

## Exercise

In [ ]:
# TASK 1: Write a Chart.yaml for a chart named 'guestbook', version 0.2.0,
#         appVersion "3.1", with at least two keywords.
# TODO: use %%writefile or cat > to create /tmp/guestbook-Chart.yaml


# TASK 2: Write a values.yaml for guestbook that includes:
#         - replicaCount (default: 2)
#         - image.repository (default: gcr.io/google-samples/gb-frontend)
#         - image.tag (default: v5)
#         - service.type (default: LoadBalancer)
#         - service.port (default: 80)
# TODO: use %%writefile or cat > to create /tmp/guestbook-values.yaml


# TASK 3: Verify your values.yaml makes sense by rendering a quick template:
#         Create a one-line ConfigMap template and run helm template on it.
# TODO:

---

## Summary

- `Chart.yaml` is the chart's identity: `name`, `version` (chart), `appVersion` (application), `type`.
- Always quote `appVersion` to prevent YAML type coercion bugs.
- `values.yaml` provides defaults; users override them with `-f` or `--set`.
- `--set` always wins over `-f`, which always wins over defaults.
- `global:` values propagate to all sub-charts — ideal for shared settings like image registry.
- Never put secrets in `values.yaml` — use Kubernetes Secrets or external secret management.

---

## Next Lesson

➡️ [Lesson 2.3 — Writing Your First Template](lesson-2.3-writing-templates.ipynb)